# 🛡️ SENTINEL — PS2 Mule-Account Detection (Colab, full pipeline)

**CyberShield Hackathon 2026 · Bank of India × IIT Hyderabad · Problem Statement 2**

Self-contained — runs from `DataSet.csv` alone. The whole engine, end to end:

1. Hygiene + **leakage blocking** (F3912 + F2230 + bucket leaks; 18 bank-listed features kept)
2. **Data Integrity Auditor** — 4-signature leak scan (our differentiator)
3. Leakage-**sensitivity sweep** — the honest-performance curve
4. **Model tournament** — LogReg / RandomForest / LightGBM / XGBoost / **CatBoost**, PR-AUC primary
5. Leakage-free **10-fold OOF** predictions (reused below)
6. **Three-way honesty validation** — bootstrap CI + in-fold leak re-check
7. Calibrate + **80/20 holdout** + threshold dial
8. **Honest improvement frontier** — recall operating point + **CV-safe graph feature**
9. **Abstention layer** — route the uncertain middle to a human
10. **Robustness probe** — graceful degradation under noise & missingness
11. **₹-cost decision curve**
12. **Mule-ring detection** — behavioral-similarity graph + validation
13. **SentinelEngine** — score + SHAP + plain-English investigation report
14. Export artifacts

> The headline isn't a 0.99 score — it's that we **detect and remove the leakage** that
> fakes a 0.99, and report a defensible **PR-AUC ≈ 0.88 (CI 0.84–0.95)**. We proved 3 ways
> that 0.99 *is* the leak: a 3-algorithm invisible-mule tail, a sensitivity plateau at ~0.86,
> and a recall ceiling of ~92.6% (97%+ recall is unreachable without the leak).

**How to use:** Runtime → Run all → upload `DataSet.csv`. CPU is fine. Full run ≈ 20–30 min;
heavy cells are flagged.


## 1 · Install dependencies

In [ ]:
!pip -q install lightgbm xgboost catboost shap scikit-learn pandas numpy networkx matplotlib joblib
import numpy as np, pandas as pd, warnings, json, time
warnings.filterwarnings("ignore")
print("ready")

## 2 · Upload `DataSet.csv`
(Or mount Drive: see the commented lines.)

In [ ]:
from google.colab import files
up = files.upload()                      # pick DataSet.csv
CSV = next(iter(up))
# from google.colab import drive; drive.mount('/content/drive'); CSV='/content/drive/MyDrive/DataSet.csv'
print("using:", CSV)

## 3 · Preprocess + leakage blocking
Drops dead/sparse cols, ordinal-encodes categoricals, adds deduped missingness flags, and
removes leakage: **F3912** (96%-aligned label proxy) + any feature with a value-bucket
>=20% fraud (~22x the 0.89% base rate). **Bank-hinted features are never auto-removed.**
(Mirrors `src/preprocess.py`.)

In [ ]:
TARGET = "F3924"
HINTS = {"F115","F321","F527","F531","F670","F1692","F2082","F2122","F2582","F2678",
         "F2737","F2956","F3043","F3836","F3887","F3889","F3891","F3894"}
LEAK_BLOCK = ["F3912"]; LEAK_RATE = 0.20; LEAK_MIN_N = 10; MIN_NONNA = 30

def detect_bucket_leaks(X, y, rate=LEAK_RATE, min_n=LEAK_MIN_N):
    leaks = []; yv = np.asarray(y)
    for c in X.columns:
        if c.split("__")[0] in HINTS: continue
        d = pd.DataFrame({"v": X[c].round(3), "y": yv}).dropna(subset=["v"])
        if d.empty: continue
        g = d.groupby("v")["y"]; cnt, mean = g.count(), g.mean(); sel = cnt >= min_n
        if sel.any() and mean[sel].max() >= rate: leaks.append(c)
    return leaks

def preprocess(csv, apply_bucket_leaks=True, verbose=True):
    df = pd.read_csv(csv, index_col=0, low_memory=False)
    y = df[TARGET].astype(int); X = df.drop(columns=[TARGET])
    X = X.drop(columns=[c for c in LEAK_BLOCK if c in X.columns])
    na = X.isna().mean(); nn = X.notna().sum()
    X = X.drop(columns=sorted(set(na[na > 0.97].index) | set(nn[nn < MIN_NONNA].index)))
    X = X.drop(columns=X.columns[X.nunique(dropna=True) <= 1])
    na2 = X.isna().mean(); flag = na2[na2 > 0.30].index
    mf = pd.DataFrame({f"{c}__ismissing": X[c].isna().astype("int8") for c in flag}, index=X.index)
    mf = mf.loc[:, ~mf.T.duplicated()]
    cat_maps = {}
    for c in X.select_dtypes(include="object").columns:
        cats = {v: i for i, v in enumerate(sorted(X[c].dropna().unique(), key=str))}
        cat_maps[c] = cats; X[c] = X[c].map(cats).astype("float32")
    X = pd.concat([X.astype("float32"), mf], axis=1)
    bl = detect_bucket_leaks(X, y) if apply_bucket_leaks else []
    X = X.drop(columns=bl)
    if verbose:
        print(f"X={X.shape}, pos={int(y.sum())} ({y.mean()*100:.2f}%), blocked F3912 + {len(bl)} bucket-leaks")
    return X, y, bl, cat_maps

X, y, bucket_leaks, CAT_MAPS = preprocess(CSV)
pw = float((y == 0).sum() / (y == 1).sum())
assert "F3912" not in X.columns and "F2230" not in X.columns
assert sum(h in X.columns for h in HINTS) == 18, "all 18 bank-listed features must survive"
print("leak-removal OK | scale_pos_weight =", round(pw, 1))

## 4 · Data Integrity Auditor — *the differentiator*
Scores every feature on 4 leak signatures (label-proxy, exact-bucket, range/decile,
univariate-AUC) and emits a severity-ranked audit any bank can run on its own data
(mirrors `src/data_integrity_auditor.py`).

In [ ]:
from sklearn.metrics import roc_auc_score
MIN_N = 10; BASE_MULT = {"CRITICAL":0.90,"HIGH":0.50,"MEDIUM":0.20,"WATCH":0.10}
def integrity_audit(csv):
    Xa, ya, _, _ = preprocess(csv, apply_bucket_leaks=False, verbose=False)
    base = float(ya.mean()); yv = ya.values; rows = []
    for col in Xa.columns:
        s = Xa[col]; nona = s.notna()
        if nona.sum() < MIN_N or ya[nona].nunique() < 2: continue
        try:    a = roc_auc_score(ya[nona], s[nona].astype(float)); auc = max(a, 1-a)
        except ValueError: auc = np.nan
        d = pd.DataFrame({"v": s.round(3), "y": yv}).dropna(subset=["v"])
        g = d.groupby("v")["y"]; cnt, mean = g.count(), g.mean(); sel = cnt >= MIN_N
        bucket = float(mean[sel].max()) if sel.any() else 0.0
        dec = 0.0
        if s.nunique() >= 10:
            try:
                q = pd.qcut(s[nona], 10, duplicates="drop"); dr = ya[nona].groupby(q, observed=True).mean()
                dn = ya[nona].groupby(q, observed=True).count(); dr = dr[dn >= MIN_N]; dec = float(dr.max()) if len(dr) else 0.0
            except (ValueError, IndexError): dec = 0.0
        worst = max(bucket, dec); cov = int(ya[nona].sum()); auc_eff = auc if (not np.isnan(auc) and cov >= 10) else 0.0
        sev = ("CRITICAL" if (worst>=0.90 or auc_eff>=0.98) else "HIGH" if (worst>=0.50 or auc_eff>=0.95)
               else "MEDIUM" if worst>=0.20 else "WATCH" if (worst>=0.10 or (not np.isnan(auc) and auc>=0.95)) else None)
        if sev:
            rows.append({"feature":col,"severity":sev,"univariate_auc":round(float(auc),3) if not np.isnan(auc) else None,
                         "best_bucket_fraud_rate":round(bucket,3),"lift_vs_base":round(worst/base,1),
                         "bank_listed":col.split("__")[0] in HINTS})
    res = pd.DataFrame(rows); od = {"CRITICAL":0,"HIGH":1,"MEDIUM":2,"WATCH":3}
    res["o"] = res["severity"].map(od)
    return res.sort_values(["o","best_bucket_fraud_rate"],ascending=[True,False]).drop(columns="o").reset_index(drop=True)
audit = integrity_audit(CSV); counts = audit["severity"].value_counts().to_dict()
print("severity:", {s: counts.get(s,0) for s in ["CRITICAL","HIGH","MEDIUM","WATCH"]})
auto_block = sorted(audit[(audit.severity.isin(["CRITICAL","HIGH"])) & (~audit.bank_listed)]["feature"])
print(f"auto-block (CRITICAL/HIGH, non-bank-listed): {len(auto_block)} features")
display(audit[audit.severity.isin(["CRITICAL","HIGH"])].head(12))

## 5 · Leakage-sensitivity sweep (the honest-performance curve)
PR-AUC as leaks are progressively removed — it **plateaus at ~0.86**, proving the remaining
signal is genuine. The steep early drop is leakage leaving. *(~2-3 min.)*

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_validate
Xf, _, _, _ = preprocess(CSV, apply_bucket_leaks=False, verbose=False)
cv3 = StratifiedKFold(3, shuffle=True, random_state=42)
def cvpr(Xm):
    m = lgb.LGBMClassifier(n_estimators=400,learning_rate=0.04,num_leaves=31,subsample=0.8,colsample_bytree=0.6,
        reg_lambda=1.0,min_child_samples=20,scale_pos_weight=pw,n_jobs=-1,random_state=42,verbose=-1)
    r = cross_validate(m,Xm,y,cv=cv3,scoring={"pr":"average_precision","roc":"roc_auc"},n_jobs=1)
    return r["test_pr"].mean(), r["test_roc"].mean()
print(f"{'thr':>6}{'n_leaks':>9}{'n_feat':>8}{'PR-AUC':>9}{'ROC':>7}")
for thr in [1.01,0.30,0.20,0.10,0.05,0.03]:
    lk = [] if thr>1 else detect_bucket_leaks(Xf,y,rate=thr); pr,roc = cvpr(Xf.drop(columns=lk))
    print(f"{('none' if thr>1 else f'{thr:.2f}'):>6}{len(lk):>9}{Xf.shape[1]-len(lk):>8}{pr:>9.3f}{roc:>7.3f}")
print(f"random baseline PR-AUC = {y.mean():.4f}")

## 6 · Model tournament (leakage-removed, PR-AUC primary)
Repeated 5x2 CV. LightGBM is the deployed headline (keeps SHAP exact). **CatBoost** (ordered
boosting) is included as an honest challenger — it tends to lead, but within the CV noise band.
*(~5-8 min.)*

In [ ]:
import xgboost as xgb, copy
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from catboost import CatBoostClassifier
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)
def cat_scores():  # CatBoost needs fillna; score it separately under the same CV
    aps = []
    for tr,va in cv.split(X,y):
        spw=float((y.iloc[tr]==0).sum()/(y.iloc[tr]==1).sum())
        cb=CatBoostClassifier(iterations=600,learning_rate=0.04,depth=6,l2_leaf_reg=4.0,boosting_type="Ordered",
            scale_pos_weight=spw,random_seed=42,verbose=0,allow_writing_files=False)
        cb.fit(X.iloc[tr].fillna(-999),y.iloc[tr])
        from sklearn.metrics import average_precision_score
        aps.append(average_precision_score(y.iloc[va], cb.predict_proba(X.iloc[va].fillna(-999))[:,1]))
    return np.array(aps)
models = {
 "LogReg": Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler()),
   ("c",LogisticRegression(max_iter=2000,C=0.1,class_weight="balanced"))]),
 "RandomForest": Pipeline([("i",SimpleImputer(strategy="median")),
   ("c",RandomForestClassifier(n_estimators=400,min_samples_leaf=2,max_features="sqrt",
   class_weight="balanced_subsample",n_jobs=-1,random_state=42))]),
 "LightGBM": lgb.LGBMClassifier(n_estimators=600,learning_rate=0.03,num_leaves=31,subsample=0.8,colsample_bytree=0.6,
   reg_lambda=1.0,min_child_samples=20,scale_pos_weight=pw,n_jobs=-1,random_state=42,verbose=-1),
 "XGBoost": xgb.XGBClassifier(n_estimators=600,learning_rate=0.03,max_depth=4,subsample=0.8,colsample_bytree=0.6,
   reg_lambda=1.0,min_child_weight=5,scale_pos_weight=pw,tree_method="hist",eval_metric="aucpr",n_jobs=-1,random_state=42)}
res = []
for n,m in models.items():
    r = cross_validate(m,X,y,cv=cv,scoring={"pr":"average_precision","roc":"roc_auc"},n_jobs=1)
    res.append({"model":n,"PR_AUC":r["test_pr"].mean(),"std":r["test_pr"].std(),"ROC_AUC":r["test_roc"].mean()})
    print(f"{n:14s} PR-AUC {res[-1]['PR_AUC']:.3f}+/-{res[-1]['std']:.3f}")
cb_ap = cat_scores(); res.append({"model":"CatBoost","PR_AUC":cb_ap.mean(),"std":cb_ap.std(),"ROC_AUC":np.nan})
print(f"{'CatBoost':14s} PR-AUC {cb_ap.mean():.3f}+/-{cb_ap.std():.3f}  (honest challenger)")
lb = pd.DataFrame(res).sort_values("PR_AUC",ascending=False).reset_index(drop=True); display(lb)
WINNER = "LightGBM"  # deployed headline keeps SHAP exact
print("deployed headline:", WINNER)
def make_winner(): return copy.deepcopy(models[WINNER])

## 7 · Leakage-free 10-fold OOF predictions (reused below)
Every account scored exactly once by a model that never saw it. Feeds the bootstrap CI,
recall operating point, abstention, ₹-cost, and ring detection. *(~1-2 min.)*

In [ ]:
skf10 = StratifiedKFold(10, shuffle=True, random_state=42); oof = np.zeros(len(y)); yv = y.to_numpy()
for tr,va in skf10.split(X,y):
    m = lgb.LGBMClassifier(n_estimators=600,learning_rate=0.03,num_leaves=31,subsample=0.8,colsample_bytree=0.6,
        reg_lambda=1.0,min_child_samples=20,scale_pos_weight=float((yv[tr]==0).sum()/(yv[tr]==1).sum()),
        n_jobs=-1,random_state=42,verbose=-1)
    m.fit(X.iloc[tr],y.iloc[tr]); oof[va] = m.predict_proba(X.iloc[va])[:,1]
from sklearn.metrics import average_precision_score, roc_auc_score
preds = pd.DataFrame({"account_id":X.index,"risk_score":np.round(100*oof).astype(int),
                      "probability":np.round(oof,4),"actual_label":yv}).sort_values("risk_score",ascending=False)
print(f"OOF PR-AUC = {average_precision_score(yv,oof):.3f} | ROC-AUC = {roc_auc_score(yv,oof):.3f}")
print(f"Top-50 watchlist: {int(preds.head(50)['actual_label'].sum())}/50 real mules")
display(preds.head(10))

## 8 · Three-way honesty validation
Is 0.88 honest or itself leak-inflated? (a) bootstrap CI over accounts (B=2000), and
(b) re-detect the bucket-leak blocklist **strictly inside each training fold**. *(~2-4 min.)*

In [ ]:
RNG = np.random.RandomState(42); n = len(yv); boots = []
for _ in range(2000):
    idx = RNG.randint(0,n,n)
    if yv[idx].sum() < 2: continue
    boots.append(average_precision_score(yv[idx], oof[idx]))
lo,hi = np.percentile(boots,[2.5,97.5])
print(f"(a) OOF PR-AUC {average_precision_score(yv,oof):.3f}  bootstrap 95% CI [{lo:.3f}, {hi:.3f}]")
Xb, yb, _, _ = preprocess(CSV, apply_bucket_leaks=False, verbose=False)
cv52 = RepeatedStratifiedKFold(5,2,random_state=42); infold = []
for tr,va in cv52.split(Xb,yb):
    lk = detect_bucket_leaks(Xb.iloc[tr], yb.iloc[tr]); keep = [c for c in Xb.columns if c not in lk]
    pwf = float((yb.iloc[tr]==0).sum()/(yb.iloc[tr]==1).sum())
    mm = lgb.LGBMClassifier(n_estimators=600,learning_rate=0.03,num_leaves=31,subsample=0.8,colsample_bytree=0.6,
        reg_lambda=1.0,min_child_samples=20,scale_pos_weight=pwf,n_jobs=-1,random_state=42,verbose=-1)
    mm.fit(Xb.iloc[tr][keep], yb.iloc[tr]); infold.append(average_precision_score(yb.iloc[va], mm.predict_proba(Xb.iloc[va][keep])[:,1]))
infold = np.array(infold)
print(f"(b) IN-FOLD PR-AUC {infold.mean():.3f} +/- {infold.std():.3f}  (headline ~0.88 -> not inflated)")

## 9 · Calibrate winner + 80/20 holdout + threshold dial

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
Xtr,Xho,ytr,yho = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
clf = CalibratedClassifierCV(make_winner(),method="sigmoid",cv=5).fit(Xtr,ytr); ph = clf.predict_proba(Xho)[:,1]
print(f"HOLDOUT PR-AUC {average_precision_score(yho,ph):.3f} | ROC {roc_auc_score(yho,ph):.3f} | Brier {brier_score_loss(yho,ph):.4f}")
tt=[]
for t in [0.3,0.5,0.7,0.9]:
    pred=ph>=t; tp=int((pred&(yho==1)).sum()); fp=int((pred&(yho==0)).sum())
    tt.append({"thr":t,"alerts":tp+fp,"mules_caught":tp,"false_alarms":fp,"precision":round(tp/(tp+fp),3) if tp+fp else 0,"recall":round(tp/yho.sum(),3)})
display(pd.DataFrame(tt))
final = CalibratedClassifierCV(make_winner(),method="sigmoid",cv=5).fit(X,y)
base_model = lgb.LGBMClassifier(n_estimators=600,learning_rate=0.03,num_leaves=31,subsample=0.8,colsample_bytree=0.6,
    reg_lambda=1.0,min_child_samples=20,scale_pos_weight=pw,n_jobs=-1,random_state=42,verbose=-1).fit(X,y)

## 10 · Honest improvement frontier (no leakage)
**How high can recall and PR-AUC honestly go?** Two truths we report plainly:

- **Recall operating point.** We pick the lowest threshold reaching a target recall and show
  the *precision cost beside it* — never recall alone. On leak-free OOF, recall **caps at ~92.6%**;
  97%+ is unreachable without the F3912 leak.
- **CV-safe graph feature.** Similarity to *training-fold* mules only (never validation labels) —
  the ring signal as a model feature, which recovers ring-embedded 'invisible' mules. Honest
  upside; PR-AUC>0.99 still needs the leak (proven by the sensitivity plateau + recall ceiling).

In [ ]:
n_pos = int(yv.sum()); rows=[]
for t in np.round(np.arange(0.01,1.0,0.01),2):
    pred=oof>=t; tp=int((pred&(yv==1)).sum()); fp=int((pred&(yv==0)).sum()); al=tp+fp
    rows.append({"threshold":float(t),"recall":tp/n_pos,"precision":(tp/al) if al else 0,"false_alarms":fp,"alerts":al})
tdf = pd.DataFrame(rows)
print("recall ladder (lowest threshold reaching each target; precision cost beside it):")
for tgt in [0.80,0.90,0.95,0.97]:
    f = tdf[tdf.recall>=tgt]
    if len(f):
        r=f.sort_values("threshold",ascending=False).iloc[0]
        print(f"  recall>={tgt:.0%}: thr={r.threshold:.2f} precision={r.precision:.3f} false_alarms={int(r.false_alarms)}")
    else:
        print(f"  recall>={tgt:.0%}: NOT ACHIEVABLE (max recall={tdf.recall.max():.1%}) -> needs the leak")
# CV-safe graph feature (mule-similarity) demonstration on OOF folds
from sklearn.metrics.pairwise import cosine_similarity
gain = pd.Series(base_model.booster_.feature_importance("gain"), index=X.columns)
topf = list(gain.sort_values(ascending=False).index[:30]); oof_g = np.zeros(len(y))
for tr,va in skf10.split(X,y):
    imp=SimpleImputer(strategy="median").fit(X.iloc[tr][topf]); sca=StandardScaler().fit(imp.transform(X.iloc[tr][topf]))
    Ztr=sca.transform(imp.transform(X.iloc[tr][topf])); mule_Z=Ztr[y.iloc[tr].values==1]
    def gf(idx):
        Z=sca.transform(imp.transform(X.iloc[idx][topf])); S=cosine_similarity(Z,mule_Z); S.sort(axis=1)
        return S[:,-min(5,S.shape[1]):].mean(axis=1)
    Xtr_g=X.iloc[tr].copy(); Xtr_g["__mule_sim"]=gf(tr); Xva_g=X.iloc[va].copy(); Xva_g["__mule_sim"]=gf(va)
    spw=float((y.iloc[tr]==0).sum()/(y.iloc[tr]==1).sum())
    m=lgb.LGBMClassifier(n_estimators=600,learning_rate=0.03,num_leaves=31,subsample=0.8,colsample_bytree=0.6,
        reg_lambda=1.0,min_child_samples=20,scale_pos_weight=spw,n_jobs=-1,random_state=42,verbose=-1).fit(Xtr_g,y.iloc[tr])
    oof_g[va]=m.predict_proba(Xva_g)[:,1]
print(f"\nbaseline OOF PR-AUC      = {average_precision_score(yv,oof):.3f}")
print(f"+ CV-safe graph feature = {average_precision_score(yv,oof_g):.3f}  (honest upside; not 0.99)")

## 11 · Abstention / uncertainty layer
Carves a **uncertain -> route to analyst** band, calibrated on OOF (mirrors `src/uncertainty.py`).

In [ ]:
TARGET_PREC, TARGET_NPV = 0.95, 0.999; grid = np.unique(np.round(oof,4)); t_hi=1.0
for t in grid:
    sel=oof>=t
    if sel.sum() and yv[sel].mean()>=TARGET_PREC: t_hi=float(t); break
t_lo=0.0
for t in grid[::-1]:
    sel=oof<=t
    if sel.sum() and (1-yv[sel]).mean()>=TARGET_NPV: t_lo=float(t); break
if t_lo>=t_hi: t_lo,t_hi=0.10,0.90
auto=(oof>=t_hi)|(oof<=t_lo); unc=~auto
err=int(((oof>=t_hi)&(yv==0)).sum()+((oof<=t_lo)&(yv==1)).sum())
print(f"t_lo={t_lo:.4f} t_hi={t_hi:.4f} | auto-decide {auto.mean():.1%} at {err/max(auto.sum(),1):.2%} error | route {unc.mean():.1%} ({int(unc.sum())}) to analysts")

## 12 · Robustness probe (noise + missingness). *(~2-3 min.)*

In [ ]:
skf5=StratifiedKFold(5,shuffle=True,random_state=42); folds=[]
for tr,va in skf5.split(X,y):
    m=lgb.LGBMClassifier(n_estimators=600,learning_rate=0.03,num_leaves=31,subsample=0.8,colsample_bytree=0.6,
        reg_lambda=1.0,min_child_samples=20,scale_pos_weight=float((yv[tr]==0).sum()/(yv[tr]==1).sum()),
        n_jobs=-1,random_state=42,verbose=-1).fit(X.iloc[tr],y.iloc[tr]); folds.append((m,va))
def scored(pf):
    o=np.zeros(len(y))
    for m,va in folds: o[va]=m.predict_proba(pf(X.iloc[va].astype("float32").copy()))[:,1]
    return average_precision_score(yv,o)
cont=[c for c in X.columns if not c.endswith("__ismissing") and c not in CAT_MAPS and X[c].nunique(dropna=True)>20]
ci=[X.columns.get_loc(c) for c in cont]; cs=X[cont].std().fillna(0).to_numpy(); nc=X.shape[1]; R=np.random.RandomState(42)
def add_noise(Xv, lv):
    if lv==0: return Xv
    a=Xv.to_numpy(copy=True); sub=a[:,ci]; mask=~np.isnan(sub)
    sub[mask]+=(R.normal(0,1,sub.shape)*(cs*lv))[mask]; a[:,ci]=sub
    return pd.DataFrame(a,index=Xv.index,columns=Xv.columns)
def drop_feats(Xv, lv):
    if lv==0: return Xv
    Xv.iloc[:, R.choice(nc,int(nc*lv),replace=False)]=np.nan; return Xv
print("NOISE x std (continuous):", {lv: round(scored(lambda Xv,lv=lv: add_noise(Xv,lv)),4) for lv in [0.0,0.1,0.25]})
print("DROPOUT (blank to NaN):", {f"{int(lv*100)}%": round(scored(lambda Xv,lv=lv: drop_feats(Xv,lv)),4) for lv in [0.0,0.1,0.25]})

## 13 · ₹-cost decision curve (editable assumptions; mirrors `src/insights.py`).

In [ ]:
AVG_MULE_LOSS=250_000; ANALYST_REVIEW_COST=400; FP_HARM_COST=5_000; rows=[]
for t in np.round(np.concatenate([[0.01,0.02,0.03],np.arange(0.05,1.0,0.05)]),2):
    pred=oof>=t; tp=int((pred&(yv==1)).sum()); fp=int((pred&(yv==0)).sum()); al=tp+fp
    rows.append({"threshold":float(t),"alerts":al,"mules_caught":tp,"recall":round(tp/int(yv.sum()),3),
                 "net_savings_rupees":int(tp*AVG_MULE_LOSS-al*ANALYST_REVIEW_COST-fp*FP_HARM_COST)})
cost=pd.DataFrame(rows); best=cost.loc[cost.net_savings_rupees.idxmax()]
print(f"optimal thr {best.threshold}: catch {int(best.mules_caught)} mules, net Rs {int(best.net_savings_rupees):,}"); display(cost[cost.threshold<=0.5])

## 14 · Mule-ring detection (behavioral-similarity graph; mirrors `src/mule_network.py`).

In [ ]:
import networkx as nx
mules=X[y==1]; M=mules[topf]
Z=StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(M)); S=cosine_similarity(Z); np.fill_diagonal(S,0)
thr=np.percentile(S[np.triu_indices_from(S,k=1)],94); ids=list(mules.index); G=nx.Graph(); G.add_nodes_from(ids)
for i in range(len(ids)):
    for j in range(i+1,len(ids)):
        if S[i,j]>=thr: G.add_edge(ids[i],ids[j])
comps=sorted([c for c in nx.connected_components(G) if len(c)>=3],key=len,reverse=True)
print(f"candidate rings(>=3)={len(comps)} | mules in rings={sum(len(c) for c in comps)} | sizes={[len(c) for c in comps]}")

## 15 · SentinelEngine — score + SHAP + investigation report (mirrors `src/sentinel.py`).

In [ ]:
import shap
BANDS=[(90,"CRITICAL"),(70,"HIGH"),(40,"MEDIUM"),(0,"LOW")]
ACTION={"CRITICAL":"Freeze outbound transfers; escalate L2; file SAR.","HIGH":"Hold high-value txns; same-day analyst review.",
        "MEDIUM":"Enhanced-monitoring watchlist.","LOW":"Routine monitoring."}
band=lambda s: next(nm for t,nm in BANDS if s>=t)
explainer=shap.TreeExplainer(base_model); cols=list(X.columns)
def report(row, aid="?"):
    Xr=row.to_frame().T.reindex(columns=cols).astype("float32"); p=float(final.predict_proba(Xr)[0,1]); s=int(round(100*p))
    tier="CONFIDENT-MULE" if p>=t_hi else "CONFIDENT-LEGIT" if p<=t_lo else "UNCERTAIN"
    sv=explainer.shap_values(Xr); sv=sv[1] if isinstance(sv,list) else sv; sv=np.asarray(sv); sv=sv[...,1] if sv.ndim==3 else sv; sv=sv.reshape(-1)
    up=[(cols[i],Xr.iloc[0,i],sv[i]) for i in np.argsort(-np.abs(sv))[:6] if sv[i]>0][:5]
    L=[f"INVESTIGATION REPORT — Account #{aid}","="*52,f"Risk {s}/100 ({band(s)}) [{tier}] | prob {p:.1%}",f"Action: {ACTION[band(s)]}","","Top risk drivers:"]
    L+=[f"  {i}. {c} = {'BLANK' if pd.isna(v) else round(float(v),3)}" for i,(c,v,_) in enumerate(up,1)]
    L+=["","F3912 (post-hoc fraud flag) excluded as leakage — behavioral signal only."]
    return "\n".join(L)
mid=preds[preds.actual_label==1].iloc[0]["account_id"]; lid=preds[preds.actual_label==0].iloc[-1]["account_id"]
print(report(X.loc[mid],mid),"\n(actual: MULE)\n"); print(report(X.loc[lid],lid),"\n(actual: LEGIT)")

## 16 · Export artifacts + predictions

> **External validation (separate, never merged into this BOI pipeline):** the method was
> also validated on BAF (external generalisation), AMLSim (graph ring recovery), IEEE-CIS +
> ULB (auditor credibility), and PaySim (behavioural) — see `artifacts/<dataset>/metrics.json`,
> each tagged with its dataset. No external rows ever enter the BOI train/eval matrices.

In [ ]:
import joblib, os
os.makedirs("artifacts",exist_ok=True); os.makedirs("outputs",exist_ok=True)
joblib.dump(final,"artifacts/sentinel_model.joblib"); joblib.dump(base_model,"artifacts/base_model.joblib")
lb.to_json("artifacts/tournament_results.json",orient="records",indent=2)
preds.to_csv("outputs/predictions.csv",index=False); preds.head(50).to_csv("outputs/top_suspicious_accounts.csv",index=False)
json.dump({"dataset":"BOI","winner":WINNER,"oof_pr_auc":float(average_precision_score(yv,oof)),
           "oof_roc_auc":float(roc_auc_score(yv,oof)),"bootstrap_pr_auc_ci95":[float(lo),float(hi)],
           "infold_pr_auc_mean":float(infold.mean()),"holdout_pr_auc":float(average_precision_score(yho,ph)),
           "graph_feature_oof_pr_auc":float(average_precision_score(yv,oof_g)),
           "abstention":{"t_lo":t_lo,"t_hi":t_hi,"coverage":float(auto.mean())},
           "bucket_leaks_removed":len(bucket_leaks),"integrity_auto_block":len(auto_block)},
          open("artifacts/run_summary.json","w"),indent=2)
!cd /content && zip -qr sentinel_artifacts.zip artifacts outputs
from google.colab import files; files.download("sentinel_artifacts.zip")
print("exported ->", os.listdir("artifacts"))